In [12]:
from qiskit.quantum_info import DensityMatrix, Kraus, partial_trace, Pauli
from qiskit.quantum_info.operators.channel.transformations import _to_kraus

In [15]:
from numpy import sqrt
from qiskit import QuantumCircuit
from qiskit.circuit.library import MCMT
from qiskit.quantum_info import Operator, DensityMatrix


qc = QuantumCircuit(5)
qc.h([1, 3, 4])
qc.append(MCMT('z', 2, 1), [3, 2, 1])
qc.x([3, 1])
qc.append(MCMT('z', 2, 1), [3, 2, 1])
qc.x([3, 1])
qc.cx([2, 4, 4, 1, 3], [0, 2, 0, 2, 0])
qc.cz(1, 0)

qc.draw('mpl')

sqrt(8)*DensityMatrix(qc).to_statevector()

Statevector([ 1.+0.00000000e+00j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              0.+0.00000000e+00j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              1.+2.15077389e-15j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              1.+2.19042471e-15j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              0.+0.00000000e+00j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
             -1.+5.38118303e-17j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              0.+0.00000000e+00j, -1.-2.15077389e-15j,  0.+0.00000000e+00j,
              1.+1.69932096e-18j,  0.+0.00000000e+00j,  0.+0.00000000e+00j,
              0.+0.00000000e+00j,  0.+0.00000000e+00j,  1.-5.38118303e-17j,
              0.+0.00000000e+00j,  1.+2.19042471e-15j,  0.+0.00000000e+00j,
              0.+0.00000000e+00j,  0.+0.00000000e+00j],
            dims=(2, 2, 2, 2, 2))


In [38]:
a = DensityMatrix([-1/sqrt(8), 0, 0, 0, 0, 0, 1/sqrt(8), 0, 0, 1/sqrt(8), 0, 0, 0, 0, 0, 1/sqrt(8),
                   0, 0, 0, -1/sqrt(8), 0, 1/sqrt(8), 0, 0, 0, 0, 1/sqrt(8), 0, 1/sqrt(8), 0, 0, 0]).to_statevector()
# DensityMatrix([1] + [0 for _ in range(31)]).evolve(Operator(a))
qc1 = QuantumCircuit(5)
qc1.x(2)
qc1.prepare_state(a, range(5))
qc1.draw('mpl')

# Operator(qc1)
DensityMatrix(qc1).to_statevector()*sqrt(8)

Statevector([ 1.00000000e+00+3.60822483e-16j,
              5.55111512e-17+1.61735257e-32j,
              6.93889390e-18+6.93889390e-18j,
             -2.87124901e-50+4.29567599e-34j,
             -1.55920851e-16-1.04083409e-17j,
              2.03836957e-33+9.44940239e-34j,
              1.00000000e+00+0.00000000e+00j,
              1.17215015e-17+5.55848928e-33j,
             -1.27569733e-16-2.86072482e-32j,
             -1.00000000e+00-2.15105711e-16j,
              4.90653893e-18-7.32881194e-34j,
             -1.30184085e-50-7.23244919e-51j,
             -1.60827390e-16+3.46944695e-18j,
             -1.59615668e-33-3.68510741e-35j,
              1.01699128e-16+2.46226605e-32j,
              1.00000000e+00+1.87350135e-16j,
              1.58948231e-16+8.67361738e-18j,
             -4.27038542e-33-1.84255370e-35j,
             -9.26330258e-17-2.23530442e-32j,
              1.00000000e+00+4.89192020e-16j,
              1.09437530e-16+2.37867147e-32j,
              1.00000000e+00+9.714

In [2]:
def damp_err(gamma, n):
    '''This method produces noise operators
    gamma [float]: Damping probability
    n [int]: Number of qubit
    '''
    
    if not isinstance(n, int) or n <= 0:
        raise ValueError(f"Number of qubit should be positive integer. Given {n}")
    if not isinstance(gamma, float) or gamma < 0 or gamma > 1:
        raise ValueError(f"Damping probability should lie between 0 and 1. Given {gamma}")
        
    from numpy import eye, zeros, kron, sqrt
    
    _E = [eye(2), zeros((2, 2))]
    _E[0][1][1] = sqrt(1-gamma)
    _E[1][0][1] = sqrt(gamma)
    
    E = list(kron(_E, eye(2**(~-n)))/sqrt(n))
    for m in range(1, n):
        E += list(kron(eye(2**m), kron(_E, eye(2**(~-n-m))))/sqrt(n))
    return E

In [32]:
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([0, 1])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [0, 1]))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([1])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [1]))
print(DensityMatrix([0, 1]).evolve(Kraus(damp_err(0.0, 1))).expand(DensityMatrix([1, 0])).measure([0])[0], partial_trace(DensityMatrix([0, 0, 1, 0]).evolve(Kraus(damp_err(0.0, 2))), [0]))
# DensityMatrix(DensityMatrix([0, 1]).data)

DensityMatrix([[0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j]],
              dims=(2,))
DensityMatrix([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
              dims=(2, 2))
01 DensityMatrix([[1.+0.j]],
              dims=())
0 DensityMatrix([[1.+0.j, 0.+0.j],
               [0.+0.j, 0.+0.j]],
              dims=(2,))
1 DensityMatrix([[0.+0.j, 0.+0.j],
               [0.+0.j, 1.+0.j]],
              dims=(2,))


In [26]:
def enc(n, k, d):
    if [n, k, d] == [5, 1, 3]:
        s0 = [0, 18, 9, 20, 10, -27, -6, -24, -29, -3, -30, -15, -17, -12, -23, 5]
        s1 = [31, 13, 22, 11, 21, -4, -25, -7, -2, -28, -1, -16, -14, -19, -8, 26]
        enc_op = zeros((2, 2**n))
        for i in s0:
            v = 0.25
            if i != abs(i):
                v = -v
            enc_op[0][abs(i)] = v
        for i in s1:
            v = 0.25
            if i != abs(i):
                v = -v
            enc_op[1][abs(i)] = v
        return Operator(transpose(enc_op))

In [27]:
from qiskit.quantum_info import Operator
from numpy import transpose, matmul, zeros

mat = enc(5, 1, 3).data
# mat
matmul(transpose(mat), mat)

array([[1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j]])

In [12]:
def choi(A):
    '''This method creates Choi representation of given matrix'''
    from numpy import matrix, concatenate, array, matmul
    vec = concatenate(array(A), axis = 0)
    V_con = [[v] for v in vec]
    return matmul(V_con, matrix(vec).conjugate())

In [13]:
choi([[0, 1.j], [1, 0]])

matrix([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 1.+0.j, 0.+1.j, 0.+0.j],
        [0.+0.j, 0.-1.j, 1.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]])

In [18]:
from qiskit.quantum_info import Choi, Operator
Choi(Operator([[0, 1.j], [1, 0]]))

Choi([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
      [0.+0.j, 1.+0.j, 0.-1.j, 0.+0.j],
      [0.+0.j, 0.+1.j, 1.+0.j, 0.+0.j],
      [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
     input_dims=(2,), output_dims=(2,))

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix, Operator

qc = QuantumCircuit(2)

DensityMatrix(qc), Operator(qc)

(DensityMatrix([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
                [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
                [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
                [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
               dims=(2, 2)),
 Operator([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
           [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
           [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
           [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j]],
          input_dims=(2, 2), output_dims=(2, 2)))